In [1]:
#Install Dependencies
!pip install xgboost scikit-learn pandas numpy joblib


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb
import joblib
import os
# Create a models directory to save outputs later
os.makedirs('models', exist_ok=True)

In [3]:
print("Loading dataset directly from GitHub...")

# URL to the raw CSV file in your GitHub repo
data_url = "https://raw.githubusercontent.com/shreyascode11/The-Last-CEO/main/corporate_ai_adoption_dataset.csv"

df = pd.read_csv(data_url)
print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")
df.head()

Loading dataset directly from GitHub...


Dataset loaded successfully with 200000 rows and 13 columns.


,company_id,industry,country,year,ai_adoption_level,ai_investment_usd,automation_rate,cost_savings,revenue_impact,productivity_gain,employee_ai_training_hours,ai_maturity_score,deployment_count
0,CORP-06613,Financial Services,China,2029,0.4987,11747237,0.4119,3519354,2751513,0.3924,76.8,6.37,29
1,CORP-01597,Agriculture,Germany,2032,0.5213,1267219,0.4580,295244,304776,0.4639,83.2,7.19,37
2,CORP-02938,Energy,United States,2024,0.6147,8168951,0.5821,2472329,5170304,0.4953,123.1,6.72,26
3,CORP-05207,Retail,Germany,2021,0.4401,1234261,0.2880,512061,-233448,0.3078,63.1,5.68,21
4,CORP-07489,Technology,United States,2024,0.1918,5000645,0.1906,2122064,2110644,0.1910,29.6,4.33,16


In [4]:
print("Engineering new features...")

# Create interaction features
df["investment_per_deployment"] = df["ai_investment_usd"] / (df["deployment_count"]+1)
df["training_per_deployment"] = df["employee_ai_training_hours"] / (df["deployment_count"]+1)
df["investment_maturity"] = df["ai_investment_usd"] * df["ai_maturity_score"]
df["automation_maturity"] = df["automation_rate"] * df["ai_maturity_score"]
df["training_adoption"] = df["employee_ai_training_hours"] * df["ai_adoption_level"]

# Define feature sets
# Note: 'cost_savings' and 'productivity_gain' are excluded from input features to prevent data leakage!
features = [
    'industry', 'country', 'year', 'ai_adoption_level', 'ai_investment_usd',
    'automation_rate', 'employee_ai_training_hours', 'ai_maturity_score', 'deployment_count',
    'investment_per_deployment', 'training_per_deployment', 'investment_maturity',
    'automation_maturity', 'training_adoption'
]

target_revenue = 'revenue_impact'
target_productivity = 'productivity_gain'

# Split X and Y variables
X = df[features]
y_revenue = df[target_revenue]
y_productivity = df[target_productivity]

Engineering new features...


In [5]:
# Identify numerical and categorical columns for the preprocessor
numeric_features = [
    'year', 'ai_adoption_level', 'ai_investment_usd', 'automation_rate',
    'employee_ai_training_hours', 'ai_maturity_score', 'deployment_count',
    'investment_per_deployment', 'training_per_deployment', 'investment_maturity',
    'automation_maturity', 'training_adoption'
]
categorical_features = ['industry', 'country']

# Build ColumnTransformer to scale numbers and OneHotEncode categories
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

print("Splitting data into train/test sets...")
X_train_rev, X_test_rev, y_train_rev, y_test_rev = train_test_split(X, y_revenue, test_size=0.2, random_state=42)
X_train_prod, X_test_prod, y_train_prod, y_test_prod = train_test_split(X, y_productivity, test_size=0.2, random_state=42)

print("Applying Preprocessor Transformations...")
# Fit & transform on training data, only transform on test data
X_train_rev_transformed = preprocessor.fit_transform(X_train_rev)
X_test_rev_transformed = preprocessor.transform(X_test_rev)

# Since we use the exact same features for productivity, we just transform it
X_train_prod_transformed = preprocessor.transform(X_train_prod)
X_test_prod_transformed = preprocessor.transform(X_test_prod)

Splitting data into train/test sets...


Applying Preprocessor Transformations...


In [6]:
print("Initializing XGBoost Models...")

# Tuned hyperparameters to decrease MAE and RMSE
xgb_params = {
    'n_estimators': 300,
    'learning_rate': 0.05,
    'max_depth': 7,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'random_state': 42,
    'n_jobs': -1
}

xgb_revenue = xgb.XGBRegressor(**xgb_params)
xgb_productivity = xgb.XGBRegressor(**xgb_params)

print("Training Revenue Impact Model...")
xgb_revenue.fit(X_train_rev_transformed, y_train_rev)

print("Training Productivity Gain Model...")
xgb_productivity.fit(X_train_prod_transformed, y_train_prod)
print("Training Complete!")

Initializing XGBoost Models...
Training Revenue Impact Model...


Training Productivity Gain Model...


Training Complete!


In [7]:
def evaluate_model(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"\n{name} Metrics:")
    print(f"  MAE:  {mae:,.2f}")
    print(f"  RMSE: {rmse:,.2f}")
    print(f"  R2:   {r2:.4f}")

# Make Predictions
y_pred_rev = xgb_revenue.predict(X_test_rev_transformed)
y_pred_prod = xgb_productivity.predict(X_test_prod_transformed)

print("--- Model Evaluation ---")
evaluate_model("Revenue Impact Model", y_test_rev, y_pred_rev)
evaluate_model("Productivity Gain Model", y_test_prod, y_pred_prod)

--- Model Evaluation ---

Revenue Impact Model Metrics:
  MAE:  1,086,151.25
  RMSE: 2,628,978.26
  R2:   0.5809

Productivity Gain Model Metrics:
  MAE:  0.03
  RMSE: 0.04
  R2:   0.9582


In [8]:
print("Saving models to 'models/' directory...")

# We save the models together with the fitted preprocessor
# so the backend can correctly format user inputs dynamically.
revenue_package = {'preprocessor': preprocessor, 'model': xgb_revenue}
productivity_package = {'preprocessor': preprocessor, 'model': xgb_productivity}

joblib.dump(revenue_package, 'models/revenue_model.joblib')
joblib.dump(productivity_package, 'models/productivity_model.joblib')

print("Models saved successfully!")

Saving models to 'models/' directory...
Models saved successfully!
